<a href="https://colab.research.google.com/github/amathe1/GenAI-AgenticAI-Hub/blob/main/Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 11.1 MB/s eta 0:00:00


In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits
import numpy as np

# Load Dataset (Example: Digits dataset)
data = load_digits()
X = data.data
y = data.target

# Preprocess the data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid = torch.Tensor(X_train), torch.Tensor(X_valid)
y_train, y_valid = torch.LongTensor(y_train), torch.LongTensor(y_valid)

train_dataset = TensorDataset(X_train, y_train)
valid_dataset = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

# Define the model architecture
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, hidden_units, dropout, activation_fn, batch_norm):
        super(MLP, self).__init__()
        layers = []

        # Input Layer
        layers.append(nn.Linear(input_dim, hidden_units))
        if batch_norm:
            layers.append(nn.BatchNorm1d(hidden_units))
        layers.append(activation_fn())
        layers.append(nn.Dropout(dropout))

        # Hidden Layers
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_units, hidden_units))
            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_units))
            layers.append(activation_fn())
            layers.append(nn.Dropout(dropout))

        # Output Layer
        layers.append(nn.Linear(hidden_units, 10))  # 10 classes in MNIST

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Define the objective function for Optuna
def objective(trial):
    # Hyperparameter tuning space
    hidden_layers = trial.suggest_int('hidden_layers', 1, 5)  # Number of hidden layers
    hidden_units = trial.suggest_int('hidden_units', 64, 256)  # Number of units per hidden layer
    dropout = trial.suggest_float('dropout', 0.2, 0.5)  # Dropout rate
    batch_norm = trial.suggest_categorical('batch_norm', [True, False])  # Use batch normalization or not

    # Activation function
    activation_fn = trial.suggest_categorical('activation_fn', [nn.ReLU, nn.LeakyReLU, nn.Sigmoid])

    # Optimizer choice
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD'])
    if optimizer_name == 'Adam':
        optimizer = optim.Adam
    else:
        optimizer = optim.SGD

    # Initialize the model
    model = MLP(input_dim=X_train.shape[1], hidden_layers=hidden_layers, hidden_units=hidden_units,
                dropout=dropout, activation_fn=activation_fn, batch_norm=batch_norm)

    # Optimizer and Loss function
    optimizer_instance = optimizer(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # Early stopping parameters
    patience = 5
    best_valid_loss = float('inf')
    counter = 0

    # Training loop
    for epoch in range(50):  # Max epochs
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer_instance.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_instance.step()
            running_loss += loss.item()

        model.eval()
        valid_loss = 0.0
        with torch.no_grad():
            for inputs, labels in valid_loader:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                valid_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        avg_valid_loss = valid_loss / len(valid_loader)

        # Early stopping
        if avg_valid_loss < best_valid_loss:
            best_valid_loss = avg_valid_loss
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

    return best_valid_loss

# Optuna study to find the best hyperparameters
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Display the best hyperparameters
print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
print(f"  Hyperparameters: {trial.params}")

# Optional: Retrain model with the best hyperparameters
best_params = trial.params
best_model = MLP(input_dim=X_train.shape[1], hidden_layers=best_params['hidden_layers'],
                 hidden_units=best_params['hidden_units'], dropout=best_params['dropout'],
                 activation_fn=best_params['activation_fn'], batch_norm=best_params['batch_norm'])

optimizer_instance = optim.Adam(best_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Final training with the best model
# Train and evaluate using best hyperparameters


[I 2025-01-31 10:29:16,920] A new study created in memory with name: no-name-3fa30110-fb9e-445d-b80b-e34374f4c7ef
/usr/local/lib/python3.11/dist-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.activation.ReLU'> which is of type type.
  warnings.warn(message)
/usr/local/lib/python3.11/dist-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.activation.LeakyReLU'> which is of type type.
  warnings.warn(message)
/usr/local/lib/python3.11/dist-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.activation.Sigmoid'> which is of type type.
  w

Best trial:
  Value: 0.020338236577420805
  Hyperparameters: {'hidden_layers': 4, 'hidden_units': 226, 'dropout': 0.30018694421569886, 'batch_norm': True, 'activation_fn': <class 'torch.nn.modules.activation.ReLU'>, 'optimizer': 'Adam'}


In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits
import numpy as np


In [ ]:
# Load Dataset (Example: Digits dataset)
data = load_digits()
data

{'data': array([[ 0.,  0.,  5., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ..., 10.,  0.,  0.],
        [ 0.,  0.,  0., ..., 16.,  9.,  0.],
        ...,
        [ 0.,  0.,  1., ...,  6.,  0.,  0.],
        [ 0.,  0.,  2., ..., 12.,  0.,  0.],
        [ 0.,  0., 10., ..., 12.,  1.,  0.]]),
 'target': array([0, 1, 2, ..., 8, 9, 8]),
 'frame': None,
 'feature_names': ['pixel_0_0',
  'pixel_0_1',
  'pixel_0_2',
  'pixel_0_3',
  'pixel_0_4',
  'pixel_0_5',
  'pixel_0_6',
  'pixel_0_7',
  'pixel_1_0',
  'pixel_1_1',
  'pixel_1_2',
  'pixel_1_3',
  'pixel_1_4',
  'pixel_1_5',
  'pixel_1_6',
  'pixel_1_7',
  'pixel_2_0',
  'pixel_2_1',
  'pixel_2_2',
  'pixel_2_3',
  'pixel_2_4',
  'pixel_2_5',
  'pixel_2_6',
  'pixel_2_7',
  'pixel_3_0',
  'pixel_3_1',
  'pixel_3_2',
  'pixel_3_3',
  'pixel_3_4',
  'pixel_3_5',
  'pixel_3_6',
  'pixel_3_7',
  'pixel_4_0',
  'pixel_4_1',
  'pixel_4_2',
  'pixel_4_3',
  'pixel_4_4',
  'pixel_4_5',
  'pixel_4_6',
  'pixel_4_7',
  'pixel_5_0',
  'pixel_5_1',
 

In [ ]:
X = data.data
y = data.target

In [ ]:
X

array([[ 0.,  0.,  5., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ..., 10.,  0.,  0.],
       [ 0.,  0.,  0., ..., 16.,  9.,  0.],
       ...,
       [ 0.,  0.,  1., ...,  6.,  0.,  0.],
       [ 0.,  0.,  2., ..., 12.,  0.,  0.],
       [ 0.,  0., 10., ..., 12.,  1.,  0.]])

In [ ]:
y

array([0, 1, 2, ..., 8, 9, 8])

In [ ]:
# Preprocess the data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid = torch.Tensor(X_train), torch.Tensor(X_valid)
y_train, y_valid = torch.LongTensor(y_train), torch.LongTensor(y_valid)

train_dataset = TensorDataset(X_train, y_train)
valid_dataset = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [ ]:
# Define the model architecture
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, hidden_units, dropout, activation_fn, batch_norm):
        super(MLP, self).__init__()
        layers = []

        # Input Layer
        layers.append(nn.Linear(input_dim, hidden_units))
        if batch_norm:
            layers.append(nn.BatchNorm1d(hidden_units))
        layers.append(activation_fn())
        layers.append(nn.Dropout(dropout))

        # Hidden Layers
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_units, hidden_units))
            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_units))
            layers.append(activation_fn())
            layers.append(nn.Dropout(dropout))

        # Output Layer
        layers.append(nn.Linear(hidden_units, 10))  # 10 classes in MNIST

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
# Define the objective function for Optuna
def objective(trial):
    # Hyperparameter tuning space
    hidden_layers = trial.suggest_int('hidden_layers', 1, 5)  # Number of hidden layers
    hidden_units = trial.suggest_int('hidden_units', 64, 256)  # Number of units per hidden layer
    dropout = trial.suggest_float('dropout', 0.2, 0.5)  # Dropout rate
    batch_norm = trial.suggest_categorical('batch_norm', [True, False])  # Use batch normalization or not

    # Activation function
    activation_fn = trial.suggest_categorical('activation_fn', [nn.ReLU, nn.LeakyReLU, nn.Sigmoid])

    # Optimizer choice
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD'])
    if optimizer_name == 'Adam':
        optimizer = optim.Adam
    else:
        optimizer = optim.SGD

    # Initialize the model
    model = MLP(input_dim=X_train.shape[1], hidden_layers=hidden_layers, hidden_units=hidden_units,
                dropout=dropout, activation_fn=activation_fn, batch_norm=batch_norm)

    # Optimizer and Loss function
    optimizer_instance = optimizer(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # Early stopping parameters
    patience = 5
    best_valid_loss = float('inf')
    counter = 0

    # Training loop
    for epoch in range(50):  # Max epochs
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer_instance.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_instance.step()
            running_loss += loss.item()

        model.eval()
        valid_loss = 0.0
        with torch.no_grad():
            for inputs, labels in valid_loader:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                valid_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        avg_valid_loss = valid_loss / len(valid_loader)

        # Early stopping
        if avg_valid_loss < best_valid_loss:
            best_valid_loss = avg_valid_loss
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

    return best_valid_loss

# Optuna study to find the best hyperparameters
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Display the best hyperparameters
print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
print(f"  Hyperparameters: {trial.params}")

[I 2026-02-07 07:48:00,404] A new study created in memory with name: no-name-227f7adb-0330-4d9a-9ed8-e56e10165602
/tmp/ipython-input-3473950266.py:10: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.activation.ReLU'> which is of type type.
  activation_fn = trial.suggest_categorical('activation_fn', [nn.ReLU, nn.LeakyReLU, nn.Sigmoid])
/tmp/ipython-input-3473950266.py:10: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.activation.LeakyReLU'> which is of type type.
  activation_fn = trial.suggest_categorical('activation_fn', [nn.ReLU, nn.LeakyReLU, nn.Sigmoid])
/tmp/ipython-input-3473950266.py:10: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modul

Best trial:
  Value: 0.02285512919964579
  Hyperparameters: {'hidden_layers': 4, 'hidden_units': 193, 'dropout': 0.2970623067951955, 'batch_norm': True, 'activation_fn': <class 'torch.nn.modules.activation.LeakyReLU'>, 'optimizer': 'Adam'}


In [ ]:
# Optional: Retrain model with the best hyperparameters
best_params = trial.params
best_model = MLP(input_dim=X_train.shape[1], hidden_layers=best_params['hidden_layers'],
                 hidden_units=best_params['hidden_units'], dropout=best_params['dropout'],
                 activation_fn=best_params['activation_fn'], batch_norm=best_params['batch_norm'])

optimizer_instance = optim.Adam(best_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()